# Task B -- fold-safe context tags with the winning recipe

The current best submission uses TAPT MuRIL, one-layer reinitialization and R-Drop.
The known error is target identification: lexical abuse cues often point at the
wrong class while the label follows the political, religious, regional or media
target. This notebook tests whether explicit context tags help MuRIL learn that
distinction.

The two matched arms are:

| arm | change |
|---|---|
| control | TAPT + one-layer reinitialization + R-Drop |
| tags | the same recipe with `--tags` |

`--tags` appends ordinary English markers such as `political topic`, `violent act
topic`, `demand mood`, `second person address` and `respectful address`. Topic
gazetteers are fitted separately inside each training fold, so validation labels do
not leak into the input text. The result is an OOF experiment, not the final
five-seed full-data submission.

Expected runtime is about 3.5--4 hours on a T4 x2 or P100. Upload with GPU and
Internet enabled, then use **Save Version -> Save & Run All**.

In [ ]:
import os, pathlib, shutil, subprocess, sys

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Build one shared TAPT checkpoint

This uses the same all-text, no-holdout TAPT settings as the current Task B
submission. The path is shared with Experiment 10 because the checkpoint recipe is
identical; an existing checkpoint is reused.

In [ ]:
TAPT_OUT = "artifacts/runs/tapt-ctxdecode"
TAPT_LOG = "artifacts/logs/context_tags_tapt.log"
if (pathlib.Path(TAPT_OUT) / "config.json").exists():
    print("using existing TAPT checkpoint:", TAPT_OUT)
else:
    run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
         "--corpus", "data/raw/multiclass_train.csv", "data/external/offenseval_kn.csv",
         "--val-frac", "0", "--min-words", "1", "--no-dedupe",
         "--epochs", "8", "--out", TAPT_OUT], log=TAPT_LOG)
assert (pathlib.Path(TAPT_OUT) / "config.json").exists(), "TAPT checkpoint was not written"
print("TAPT checkpoint ready:", TAPT_OUT)

## 2. Train the matched five-fold arms

Both arms use the same deduplicated rows, five-fold split, seed 42, six epochs,
one reinitialized layer, R-Drop 0.5, FGM, EMA, class weighting and last-checkpoint
selection. The only intended difference is `--tags`.

In [ ]:
COMMON = [
    "--model", TAPT_OUT,
    "--folds", "5",
    "--reinit-layers", "1",
    "--rdrop", "0.5",
    "--aux-weight", "0",
    "--seeds", "42",
    "--epochs", "6",
    "--select", "last",
]
ARMS = [
    ("b_context_control_5f", ["--no-tags"], False),
    ("b_context_tags_5f", ["--tags"], True),
]
for tag, extra, use_tags in ARMS:
    run([sys.executable, "-u", "-m", "hastika.task_b.train",
         "--tag", tag, *COMMON, *extra],
        log=f"artifacts/logs/{tag}.log")
    run_dir = pathlib.Path("artifacts/runs") / tag
    assert (run_dir / "oof_probs.npy").exists(), f"missing OOF output for {tag}"
    assert (run_dir / "test_probs.npy").exists(), f"missing test output for {tag}"
    assert (run_dir / "predictions.csv").exists(), f"missing predictions for {tag}"
    log_text = pathlib.Path("artifacts/logs") / f"{tag}.log"
    expected = f"tags={use_tags}"
    assert expected in log_text.read_text(), f"{tag} did not run with {expected}"
print("both matched context arms completed")

## 3. Compare OOF macro-F1 and class-level effects

The tags arm is useful only if it improves the same OOF rows under the same split.
The cell reports macro-F1, accuracy, every class F1, and the change from the control.
This is the decision point for whether to build a five-seed full-data tags submission.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score
from hastika.common.preprocessing import dedupe_index

LABELS = ["Gender", "Geo-political", "Others", "Political", "Religion", "Violence"]
train = pd.read_csv("data/raw/multiclass_train.csv")
keep = dedupe_index(train["Comment"].tolist(), train["Hate Category"].tolist(), "task B")
train = train.iloc[keep].reset_index(drop=True)
y = train["Hate Category"].map(LABELS.index).to_numpy()

records, probabilities = [], {}
for tag, _, use_tags in ARMS:
    probs = np.load(pathlib.Path("artifacts/runs") / tag / "oof_probs.npy")
    assert probs.shape == (len(y), len(LABELS)), f"unexpected OOF shape for {tag}: {probs.shape}"
    probabilities[use_tags] = probs
    pred = probs.argmax(1)
    report = classification_report(y, pred, labels=range(len(LABELS)),
                                  target_names=LABELS, output_dict=True, zero_division=0)
    records.append({"arm": "tags" if use_tags else "control",
                    "macro_f1": f1_score(y, pred, average="macro"),
                    "accuracy": accuracy_score(y, pred),
                    **{f"f1_{label}": report[label]["f1-score"] for label in LABELS}})

scores = pd.DataFrame(records).set_index("arm").loc[["control", "tags"]]
print(scores.round(4).to_string())
delta = scores.loc["tags"] - scores.loc["control"]
print("\nTags minus control:")
print(delta.round(4).to_string())
print(f"\nDecision: {'tags improves' if delta['macro_f1'] > 0 else 'keep the control'} the OOF macro-F1.")

print("\nPer-class reports:")
for use_tags, probs in probabilities.items():
    print(f"\n{'tags' if use_tags else 'control'}")
    print(classification_report(y, probs.argmax(1), labels=range(len(LABELS)),
                                target_names=LABELS, digits=3, zero_division=0))

## 4. Preserve reproducibility outputs

Download this folder from Kaggle. The OOF matrices are reusable for further
analysis, while the tags arm should be trained on all labelled rows and averaged
over five seeds before being considered for CodaBench.

In [ ]:
OUT = pathlib.Path("/kaggle/working/context_tags_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for tag, _, _ in ARMS:
    run_dir = pathlib.Path("artifacts/runs") / tag
    for name in ["oof_probs.npy", "test_probs.npy", "predictions.csv"]:
        shutil.copy2(run_dir / name, OUT / f"{tag}_{name}")
    shutil.copy2(pathlib.Path("artifacts/logs") / f"{tag}.log", OUT / f"{tag}.log")
if pathlib.Path(TAPT_LOG).exists():
    shutil.copy2(TAPT_LOG, OUT / pathlib.Path(TAPT_LOG).name)
scores.to_csv(OUT / "scores.csv")
print("download:", OUT)
print("files:", sorted(x.name for x in OUT.iterdir()))